# 02 · Modelagem inicial e avaliação em desenvolvimento | CRISP-DM

Lucas Menghi desenvolve esta modelagem com assistência de agente e aprovação humana no fluxo. Esta execução é experimental e não constitui aprovação final de modelo.

## 1. Protocolo definido antes dos resultados

Comparação de quatro baselines (zero, média do treino, gasto em 30 dias e gasto em 90 dias / 3) com três regressões. Usamos cinco validações mensais de junho a outubro; treino expansivo com a variável resposta madura antes de cada referência.

Parâmetros fixos, sem busca automática: Ridge `alpha=100`; Tweedie `power=1.5, alpha=10`; boosting com 150 iterações, 15 folhas e regularização 10. São pontos iniciais de comparação, não configurações ótimas. Escala é ajustada somente no treino. Boosting não usa early stopping aleatório. Previsões negativas são truncadas em zero e contabilizadas.

RMSE em £ é a referência inicial para valor esperado. MAE mostra erro típico; viés positivo significa superestimação. Métricas são reportadas por período, agregadas e por segmentos. Clientes repetidos e janelas compartilhadas impedem tratar observações como independentes.

Novembro continua reservado. Não treinamos nem publicamos um modelo final.

> Registro histórico desta etapa. O estado final, incluindo a avaliação de novembro, está em [03_final_model.ipynb](03_final_model.ipynb).

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display
ROOT = Path.cwd() if (Path.cwd() / "src/retail.py").exists() else Path.cwd() / "Online Retail"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.retail import (locate_data, read_transactions, candidate_sales, development_table,
                        FEATURES, RESPONSE, TEST_REFERENCE, temporal_fold)
pd.set_option("display.max_columns", 25)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
(ROOT / "reports").mkdir(exist_ok=True)
(ROOT / "data/processed").mkdir(parents=True, exist_ok=True)

import json
import hashlib
import platform
from importlib.metadata import version
from datetime import datetime, timezone
from src.evaluation import evaluate_development, metrics
table = development_table(read_transactions(locate_data(ROOT)))
detail, comparison, predictions = evaluate_development(table)
detail.to_csv(ROOT / "reports/backtest_metrics.csv", index=False)
comparison.to_csv(ROOT / "reports/model_comparison.csv", index=False)
display(comparison)
display(detail.pivot(index="reference_date", columns="model", values="rmse_gbp"))
display(detail[["reference_date", "model", "total_bias_pct", "negative_predictions_clipped"]])

,model,n,rmse_gbp,mae_gbp,bias_gbp,observed_total_gbp,predicted_total_gbp,total_bias_pct
2,last_90d_div3,10027,"1,264.07",298.19,35.91,"2,795,752.38","3,155,853.79",12.88
0,hist_gradient_boosting,10027,"1,283.35",283.63,-36.77,"2,795,752.38","2,427,040.13",-13.19
3,ridge,10027,"1,342.68",282.06,-23.75,"2,795,752.38","2,557,584.67",-8.52
1,last_30d,10027,"1,630.30",361.40,70.15,"2,795,752.38","3,499,110.19",25.16
4,train_mean,10027,"1,645.33",369.46,-45.96,"2,795,752.38","2,334,875.55",-16.48
6,zero,10027,"1,668.30",278.82,-278.82,"2,795,752.38",0.00,-100.00
5,tweedie,10027,"22,100.34",881.32,552.14,"2,795,752.38","8,332,046.08",198.03


model,hist_gradient_boosting,last_30d,last_90d_div3,ridge,train_mean,tweedie,zero
reference_date,,,,,,,
2011-06-01,"1,212.74",961.49,"1,036.12","1,295.78","1,422.42","1,995.03","1,442.31"
2011-07-01,710.86,"1,369.75",745.38,873.63,911.78,"24,151.07",938.28
2011-08-01,"1,117.12","1,524.17","1,192.37","1,227.95","1,398.77","22,517.87","1,420.32"
2011-09-01,"1,861.99","2,097.32","1,797.17","1,845.98","2,335.13","7,657.36","2,358.64"
2011-10-01,"1,265.35","1,926.36","1,317.68","1,307.69","1,816.29","34,777.86","1,844.20"


,reference_date,model,total_bias_pct,negative_predictions_clipped
0,2011-06-01,zero,-100.00,0
1,2011-06-01,train_mean,-5.60,0
2,2011-06-01,last_30d,43.39,0
3,2011-06-01,last_90d_div3,21.05,0
4,2011-06-01,ridge,-12.19,21
5,2011-06-01,tweedie,5.28,0
6,2011-06-01,hist_gradient_boosting,-11.03,13
7,2011-07-01,zero,-100.00,0
8,2011-07-01,train_mean,3.49,0
9,2011-07-01,last_30d,51.56,0


## 2. Erros por segmento

A separação entre resposta zero e positiva explica onde o modelo erra; não é uma regra de seleção disponível antecipadamente. Faixas de gasto histórico usam apenas informação passada e limites fixos para descrição, não para otimizar resultados.

Uma redução no erro agregado pode esconder piora em períodos ou clientes valiosos. Não estabelecemos um vencedor apenas pela primeira linha do ranking.

In [2]:
predictions["repurchase"] = np.where(predictions[RESPONSE].eq(0), "zero", "positive")
predictions["historical_spend_band"] = pd.cut(predictions.spend_90d, bins=[-np.inf, 500, 2000, np.inf],
                                             labels=["up_to_500", "500_to_2000", "above_2000"])
segments = []
for dimension in ["repurchase", "historical_spend_band"]:
    for (model, segment), group in predictions.groupby(["model", dimension], observed=True):
        segments.append({"model": model, "dimension": dimension, "segment": str(segment),
                         **metrics(group[RESPONSE], group.prediction)})
segment_metrics = pd.DataFrame(segments)
segment_metrics.to_csv(ROOT / "reports/segment_metrics.csv", index=False)
display(segment_metrics)

manifest = {"executed_at_utc": datetime.now(timezone.utc).isoformat(),
            "data_sha256": hashlib.sha256(locate_data(ROOT).read_bytes()).hexdigest(),
            "python": platform.python_version(),
            "packages": {p: version(p) for p in ["numpy", "pandas", "scikit-learn", "nbformat", "nbclient", "matplotlib"]},
            "features": FEATURES, "response": RESPONSE, "lookback_days": 90, "horizon_days": 30,
            "rules": {"deduplicate": False, "merchandise_only": False, "net_of_returns": False},
            "model_parameters": {"ridge": {"alpha": 100}, "tweedie": {"power": 1.5, "alpha": 10, "max_iter": 2000},
                                 "hist_gradient_boosting": {"max_iter": 150, "max_leaf_nodes": 15,
                                 "min_samples_leaf": 30, "l2_regularization": 10, "early_stopping": False, "random_state": 42}},
            "training_features_scaled": ["ridge", "tweedie"], "prediction_floor": 0,
            "validation_references": [pd.Timestamp(t).strftime("%Y-%m-%d") for t in sorted(detail.reference_date.unique())],
            "test_reference": "2011-11-01", "test_evaluated": False,
            "business_rules_final_approval": "pending", "production_approval": "pending"}
(ROOT / "reports/run_manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")

,model,dimension,segment,n,rmse_gbp,mae_gbp,bias_gbp,observed_total_gbp,predicted_total_gbp,total_bias_pct
0,hist_gradient_boosting,repurchase,positive,3381,"2,159.56",559.04,-391.18,"2,795,752.38","1,473,181.38",-47.31
1,hist_gradient_boosting,repurchase,zero,6646,335.12,143.52,143.52,0.00,"953,858.75",NaN
2,last_30d,repurchase,positive,3381,"2,588.98",633.34,-230.43,"2,795,752.38","2,016,673.96",-27.87
3,last_30d,repurchase,zero,6646,774.65,223.06,223.06,0.00,"1,482,436.23",NaN
4,last_90d_div3,repurchase,positive,3381,"2,080.21",497.41,-280.43,"2,795,752.38","1,847,611.83",-33.91
5,last_90d_div3,repurchase,zero,6646,457.55,196.85,196.85,0.00,"1,308,241.96",NaN
6,ridge,repurchase,positive,3381,"2,244.32",555.53,-351.41,"2,795,752.38","1,607,649.88",-42.50
7,ridge,repurchase,zero,6646,396.85,142.93,142.93,0.00,"949,934.78",NaN
8,train_mean,repurchase,positive,3381,"2,814.58",638.25,-593.76,"2,795,752.38","788,249.22",-71.81
9,train_mean,repurchase,zero,6646,232.87,232.72,232.72,0.00,"1,546,626.32",NaN


1609

## 3. Interpretação e próximo ciclo

Leia o ranking junto dos folds e segmentos. Melhor RMSE no desenvolvimento não comprova superioridade futura. O conjunto foi usado para exploração e comparação; estes resultados não são avaliação final independente.

Antes do refinamento, revise a regra da variável resposta à luz dos cancelamentos e da sensibilidade. Depois compare regularização e complexidade em validação temporal, documentando todas as tentativas. A aprovação humana da configuração e das regras precede a abertura do teste final.

Referências: [TweedieRegressor](https://scikit-learn.org/1.2/modules/generated/sklearn.linear_model.TweedieRegressor.html), [HistGradientBoostingRegressor](https://scikit-learn.org/1.2/modules/generated/sklearn.ensemble.HistGradientBoostingRegressor.html).

## 4. Leitura desta execu??o

Nos cinco folds, a regra **gasto em 90 dias / 3** obteve o menor RMSE agregado (?1.264,07). O boosting ficou em ?1.283,35 e Ridge em ?1.342,68. Portanto, o experimento inicial ainda n?o demonstrou ganho de RMSE com ML.

O baseline zero teve MAE de ?278,82, mas vi?s de ?100%: ilustra o conflito entre minimizar erro absoluto e estimar receita esperada. Tweedie, nesta configura??o, apresentou RMSE de ?22.100,34; a converg?ncia sem avisos n?o impediu previs?es inadequadas.

Esses valores pertencem ? execu??o identificada no manifesto; se os dados ou par?metros mudarem, revisar esta interpreta??o junto das novas sa?das. Veja [revis?o detalhada](docs/development_review.md).

**Pr?ximo ciclo:** revisar a defini??o comercial da vari?vel resposta, diagnosticar os maiores erros e refinar as regress?es sob valida??o temporal. Novembro permanece reservado.